In [ ]:
import time
from dataclasses import dataclass
from typing import Dict, Any, List

import ollama  # pip install ollama

In [2]:
MODELS = ["phi3:mini", "orca-mini:3b", "gemma3:4b", "qwen3:0.6b", "deepseek-r1:1.5b"]

In [ ]:
@dataclass
class RunResult:
    model: str
    ok: bool
    latency_s: float
    prompt_eval_count: int | None
    eval_count: int | None
    total_duration_s: float | None
    answer: str | None
    error: str | None

In [ ]:
def _sync_chat(
    model: str,
    prompt: str,
    system: str | None = None,
    **opts
) -> Dict[str, Any]:

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    resp = ollama.chat(
        model=model,
        messages=messages,
        options=opts or None,
    )
    return resp


In [ ]:
from typing import Dict, Any, Tuple
def call_model_with_timing(
    model: str,
    prompt: str,
    system: str | None = None,
    **opts
) -> Tuple[Dict[str, Any], float]:
    """
    역할:
        - Ollama 모델을 동기 호출(_sync_chat)
        - 호출에 걸린 시간을 측정해서 resp와 latency(초)를 반환
    """
    t0 = time.perf_counter()
    resp = _sync_chat(model, prompt, system, **opts)  # 여기서 실제 모델 호출
    t1 = time.perf_counter()
    latency_s = t1 - t0
    return resp, latency_s

In [ ]:
def benchmark(
    prompts: List[str],
    system: str | None = None,
    **opts
) -> List[RunResult]:

    results: List[RunResult] = []

    for p in prompts:
        for m in MODELS:
            print(f"\n>>> Running model={m}")
            r = run_one(m, p, system, **opts)
            results.append(r)

    return results

In [ ]:
def parse_ollama_response(resp: Dict[str, Any]) -> Dict[str, Any]:
    """
    역할:
        - Ollama 응답 dict에서 우리가 비교에 쓸 핵심 값만 뽑아 정리
        - 모델/버전에 따라 없는 필드가 있을 수 있어 .get 사용
    반환:
        {
            "answer": str,
            "prompt_eval_count": int|None,
            "eval_count": int|None,
            "total_duration_s": float|None
        }
    """
    answer = resp.get("message", {}).get("content", "")

    prompt_eval_count = resp.get("prompt_eval_count")
    eval_count = resp.get("eval_count")

    total_duration_ns = resp.get("total_duration")
    total_duration_s = (
        total_duration_ns / 1e9
        if isinstance(total_duration_ns, (int, float))
        else None
    )

    return {
        "answer": answer,
        "prompt_eval_count": prompt_eval_count,
        "eval_count": eval_count,
        "total_duration_s": total_duration_s,
    }

In [ ]:
def print_table(results: List[RunResult]):
    print("\n=== RESULTS ===")
    for r in results:
        if r.ok:
            print(
                f"[OK] {r.model:18} "
                f"latency={r.latency_s:.2f}s  "
                f"prompt_toks={r.prompt_eval_count}  "
                f"gen_toks={r.eval_count}  "
                f"ollama_total={r.total_duration_s:.2f}s"
                if r.total_duration_s
                else f"[OK] {r.model:18} latency={r.latency_s:.2f}s"
            )
        else:
            print(f"[ERR] {r.model:18} latency={r.latency_s:.2f}s  error={r.error}")

In [ ]:
def build_run_result(
    *,
    model: str,
    ok: bool,
    latency_s: float,
    parsed: Dict[str, Any] | None = None,
    error: Exception | None = None
) -> RunResult:
    """
    역할:
        - 성공/실패 케이스 모두 RunResult 형태로 통일해서 반환
        - parsed는 parse_ollama_response() 결과(성공일 때만)
        - error는 예외 객체(실패일 때만)
    """
    if ok:
        assert parsed is not None, "ok=True이면 parsed가 필요합니다."
        return RunResult(
            model=model,
            ok=True,
            latency_s=latency_s,
            prompt_eval_count=parsed.get("prompt_eval_count"),
            eval_count=parsed.get("eval_count"),
            total_duration_s=parsed.get("total_duration_s"),
            answer=parsed.get("answer"),
            error=None,
        )

    # 실패 케이스
    return RunResult(
        model=model,
        ok=False,
        latency_s=latency_s,
        prompt_eval_count=None,
        eval_count=None,
        total_duration_s=None,
        answer=None,
        error=repr(error) if error else "UnknownError",
    )

In [ ]:
def run_one(
    model: str,
    prompt: str,
    system: str | None = None,
    **opts
) -> RunResult:
    """
    역할(한 문장):
        - '모델 호출 → 응답 파싱 → 결과 포장'을 순서대로 수행하는 오케스트레이터
    """
    try:
        resp, latency_s = call_model_with_timing(model, prompt, system, **opts)
        parsed = parse_ollama_response(resp)
        return build_run_result(model=model, ok=True, latency_s=latency_s, parsed=parsed)

    except Exception as e:
        # 실패 시에도 latency는 남기는 게 교육적으로/실험적으로 유용해서 측정
        # (호출 전에 터졌다면 거의 0에 가까운 값이 나올 수 있음)
        latency_s = 0.0
        return build_run_result(model=model, ok=False, latency_s=latency_s, error=e)

In [ ]:
def predict_main():
    prompts = [
        "다음 파이썬 함수를 버그 없이 고쳐줘:\n\ndef add(a,b):\n return a- b",
        "ROS2에서 nav2 controller_server 역할을 5줄로 설명해줘.",
        "아래 문장을 영어로 자연스럽게 번역해줘: '이번 주에 데모 가능할까요?'",
    ]

    system = "너는 간결하고 정확한 한국어 기술 도우미야. 중간과정은 설명하지마"

    opts = dict(
        temperature=0.2,
        num_predict=256,  # 공정 비교 핵심
    )

    results = benchmark(prompts, system=system, **opts)
    
    return results


In [12]:
results = predict_main()


>>> Running model=phi3:mini

>>> Running model=orca-mini:3b

>>> Running model=gemma3:4b

>>> Running model=qwen3:0.6b

>>> Running model=deepseek-r1:1.5b

>>> Running model=phi3:mini

>>> Running model=orca-mini:3b

>>> Running model=gemma3:4b

>>> Running model=qwen3:0.6b

>>> Running model=deepseek-r1:1.5b

>>> Running model=phi3:mini

>>> Running model=orca-mini:3b

>>> Running model=gemma3:4b

>>> Running model=qwen3:0.6b

>>> Running model=deepseek-r1:1.5b


In [13]:
results

[RunResult(model='phi3:mini', ok=True, latency_s=2.531516000002739, prompt_eval_count=116, eval_count=106, total_duration_s=2.527892083, answer='아래에 정의된 버그가 있는 Python 함수를 빠르게 수정하여 보완한다. 매개변수 `a`과 `b`를 합산하도록 수정해주세요.\n\n```python\ndef add(a, b):\n    return a + b\n```', error=None),
 RunResult(model='orca-mini:3b', ok=True, latency_s=3.4446496660093544, prompt_eval_count=179, eval_count=128, total_duration_s=3.443355208, answer=' 저 사진 파이썬 함수를 버그 없이 고쳐줘입니다. \n\nThis code defines a function called "add" that takes two arguments, and returns the result of subtracting the first argument from the second argument. \n\nIn other words, if you call the function with any two numbers as arguments, it will return the difference between those two numbers.', error=None),
 RunResult(model='gemma3:4b', ok=True, latency_s=2.303138499992201, prompt_eval_count=62, eval_count=19, total_duration_s=2.302166708, answer='```python\ndef add(a, b):\n  return a + b\n```', error=None),
 RunResult(model='qwen3:0.6b'

In [14]:
# 응답 본문 출력
for r in results:
    if r.ok:
        print("\n" + "=" * 80)
        print(f"MODEL: {r.model}")
        print(r.answer[:1200])


MODEL: phi3:mini
아래에 정의된 버그가 있는 Python 함수를 빠르게 수정하여 보완한다. 매개변수 `a`과 `b`를 합산하도록 수정해주세요.

```python
def add(a, b):
    return a + b
```

MODEL: orca-mini:3b
 저 사진 파이썬 함수를 버그 없이 고쳐줘입니다. 

This code defines a function called "add" that takes two arguments, and returns the result of subtracting the first argument from the second argument. 

In other words, if you call the function with any two numbers as arguments, it will return the difference between those two numbers.

MODEL: gemma3:4b
```python
def add(a, b):
  return a + b
```

MODEL: qwen3:0.6b


MODEL: deepseek-r1:1.5b


MODEL: phi3:mini
ROS2의 `controller_server`는 자동차 안전도시를 위한 주행 제어 서버로, 이 역할을 수행하기 위해서는 다음과 같은 5줄의 설명이 필요합니다:

1. `controller_server`는 nav2에서 자동차 주행을 관리하기 위해 ROS2를 사용하며, 이 서버는 주행 전반적인 방식을 제어합니다.
2. 주행 중에서 자동차가 관속을 수정하기 위해 `controller_server`는 주행 관람자의 요구를 인식하고, 이를 기반으로 주행 전반의 방

MODEL: orca-mini:3b
 Sure, I can help explain the role of the `nav2` and `controller_server` in ROS2. 

The `nav2` is a node type that provides 

In [15]:
import ollama

resp = ollama.chat(
    model="qwen3:0.6b",
    messages=[
        {"role": "system", "content": "너는 한국어를 잘 이해하는 로봇 보조 비서야. 답변은 간결하게 해."},
        {"role": "user", "content": "아래 내용을 회의록으로 정리해줘.\n- 다음 주 로봇 데모가 있다\n- 배터리 점검이 필요하다\n- 영상 촬영도 진행한다"}
    ],
    options={
        "temperature": 0.2,
        "num_predict": 256
    }
)

print(resp["message"]["content"])

- 다음 주 로봇 데모 진행  
- 배터리 점검 필요  
- 영상 촬영 진행  

(요약 완성)


In [17]:
resp = ollama.chat(
    model="qwen3:0.6b",
    messages=[
        {"role": "system", "content": "너는 한국어를 잘 이해하는 로봇 보조 비서야. 답변은 간결하게 해."},
        {"role": "user", "content": "아래 내용을 JSON 형태로 정리해줘. \n - 항목: LiDAR 점검, 중요도: 높음 \n- 항목: 카메라 캘리브레이션, 중요도: 중간 \n- 항목: 배터리 충전, 중요도: 높음"}
    ],
    options={
        "temperature": 0.2,
        "num_predict": 256
    }
)

print(resp["message"]["content"])

```json
[
  {
    "항목": "LiDAR 점검",
    " 중요도": "높음"
  },
  {
    "항목": "카메라 캘리브레이션",
    " 중요도": "중간"
  },
  {
    "항목": "배터리 충전",
    " 중요도": "높음"
  }
]
```


In [18]:
prompt = """다음 조건을 만족하는 “로봇 데모 당일 운영 계획”을 만들어줘.

- 오전 10시~12시: 리허설(자율주행 2회, 팔 조작 2회)
- 오후 1시~3시: 관람객 데모(자율주행 4회, 팔 조작 3회)
- 각 데모는 12분, 데모 사이 휴식 6분
- 배터리 교체는 15분이 필요하고, 2회 이상 해야 함
- 촬영팀이 2시~2시30분에만 와서 이 시간에는 “팔 조작”이 반드시 1회 포함되어야 함
- 결과는 타임테이블 표 + 체크리스트로 정리해줘
- 그리고 한글로 답해줘
"""

In [19]:
resp = ollama.chat(
    model="phi3:mini",
    messages=[
        {"role": "system", "content": "너는 로봇 데모 운영을 잘하는 실무 엔지니어야. 답은 표와 체크리스트 중심으로"},
        {"role": "user", "content": prompt},
    ],
    options={
        "temperature": 0.3,
        "num_predict": 1024
    }
)

print(resp["message"]["content"])

로봇 데모 당일 운영 계획:

| 시간 | 프로토타인 | 자율주행 | 팔 조작 | 휴식 | 배터리 교체 |
|------|-----------|----------|---------|------|--------------|
| 10:00 - 12:0 end | 자율주행 2회 | X        |         |       |               |
| 12:05 - 13:05 | 자율주행 4회 |          | X       | 6:00 |               |
| 13:10 - 13:15 |           |          |         |  6:00 |     15:00     |
| 13:20 - 13:45 | 자율주행 2회 | X        |         |       |               |
| 13:50 - 14:50 | 자율주행 2회 |          |         |  6:00 |     15:00     |
| 14:55 - 15:00 |           |          | X       |       |               |
| 15:05 - 17:0 end | 자율주행 2회 |          |         |  6:00 |     15:00     |
| 17:05 - 18:0 end | 자율주행 2회 |          | X       |       |               |
| 18:05 - 19:0 end | 자율주행 2회 |          |         |  6:00 |     15:00     |
| 19:05 - 20:0 end | 자율주행 4회 | X        |         |       |               |
| 20:05 - 20:30 |           |          | X       |  6:00 |     15:00     |

체크리스트:

- 자율주행: 10시~12시, 13:20~13:45, 17:05~18:0시, 1

로봇 데모 당일 운영 계획:

| 시간 | 프로토타인 | 자율주행 | 팔 조작 | 휴식 | 배터리 교체 |
|------|-----------|----------|---------|------|--------------|
| 10:00 - 12:0 end | 자율주행 2회 | X        |         |       |               |
| 12:05 - 13:05 | 자율주행 4회 |          | X       | 6:00 |               |
| 13:10 - 13:15 |           |          |         |  6:00 |     15:00     |
| 13:20 - 13:45 | 자율주행 2회 | X        |         |       |               |
| 13:50 - 14:50 | 자율주행 2회 |          |         |  6:00 |     15:00     |
| 14:55 - 15:00 |           |          | X       |       |               |
| 15:05 - 17:0 end | 자율주행 2회 |          |         |  6:00 |     15:00     |
| 17:05 - 18:0 end | 자율주행 2회 |          | X       |       |               |
| 18:05 - 19:0 end | 자율주행 2회 |          |         |  6:00 |     15:00     |
| 19:05 - 20:0 end | 자율주행 4회 | X        |         |       |               |
| 20:05 - 20:30 |           |          | X       |  6:00 |     15:00     |

체크리스트:

- 자율주행: 10시~12시, 13:20~13:45, 17:05~18:0시, 19:05~20:0시, 20:05~20:30
- 팔 조작: 12시10분~12시50분, 14시50분~15시00분, 18시00분~18시30분, 20시00분~20시05분
- 휴식: 13:10~13:15, 15:05~15:30, 17:05~17:45, 19:05~19:45
- 배터리 교체: 13:20~13:35, 18:05~18:20
